# Tutorial 00 — Setup & Environment

Companion explainer: **00_setup_environment.md**. This notebook installs the
biosphere + LCIA methods into a project and sanity-checks the install. Run it
once; later tutorials assume the `bw25-tutorials` project exists.

In [1]:
import sys
import bw2data as bd
import bw2calc as bc
import bw2io as bi

print("Python  ", sys.version.split()[0])
print("bw2data ", bd.__version__)
print("bw2calc ", bc.__version__)
print("bw2io   ", bi.__version__)

13:14:56-0400

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

...

4.7

bw2calc 

2.5.0

bw2io   

0.9.17

## Where Brightway stores data
Everything lives in a per-user data directory, split into isolated **projects**.

In [2]:
print("data dir:", bd.projects.dir)
print("existing projects:", [p.name for p in bd.projects])

data dir:

C:\Users\derne\AppData\Local\pylca\Brightway3\default.c21f969b

existing projects:

['default', 'bw25-tutorials', 'cs1-bottle', 'cs2-biofuel', 'cs3-cement', 'bw-mcp-test', 'bw-mcp-demo', 'jsonld-lcia-test']

## Create/activate the tutorial project + install biosphere & methods

On this stack (bw2data 4.7 / bw2io 0.9.17) the classic `bw2io.bw2setup()`
errors while writing methods, so we use the officially-recommended **remote
prepared project**, which ships elementary flows *and* LCIA methods, free of
any ecoinvent license.

In [3]:
PROJECT = "bw25-tutorials"
if PROJECT not in bd.projects:
    bi.remote.install_project("ecoinvent-3.10-biosphere", PROJECT)
bd.projects.set_current(PROJECT)
print("active project:", bd.projects.current)
print("databases:", list(bd.databases))

active project:

bw25-tutorials

databases:

['ecoinvent-3.10-biosphere', 'smoke_widget', 'smoke_incr', 't03_kettle', 't05_kettle', 't01_widget', 't04_kettle_xl', 't06_kettle', 't07_kettle', 't08_param', 't08_sweep', 't09_cups', 't10_sys']

## The biosphere-name quirk
The prepared project names the biosphere DB `ecoinvent-3.10-biosphere`, **not**
`biosphere3`. Always resolve it dynamically:

In [4]:
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)
print("biosphere database:", BIOSPHERE)
print("elementary flows:", len(bio))
print("LCIA methods:", len(bd.methods))

biosphere database:

ecoinvent-3.10-biosphere

elementary flows:

4362

LCIA methods:

669

## Sanity checks: find CO2, find an IPCC GWP100 method

In [5]:
co2 = next(f for f in bio
           if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",))
print("found flow:", co2["name"], co2["categories"], "| unit:", co2.get("unit"))

gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
print("GWP method:", gwp)
print("method unit:", bd.Method(gwp).metadata.get("unit"))

found flow:

Carbon dioxide, fossil

('air',)

| unit:

kilogram

GWP method:

('IPCC 2013', 'climate change', 'global warming potential (GWP100)')

method unit:

kg CO2-Eq

A few example methods across families (there are hundreds):

In [6]:
import itertools
for fam in ("IPCC 2013", "ReCiPe 2016", "EF v3"):
    hits = [m for m in bd.methods if fam in str(m)]
    print(f"{fam}: {len(hits)} methods, e.g. {hits[0] if hits else '—'}")

IPCC 2013: 8 methods, e.g. ('IPCC 2013 no LT', 'climate change no LT', 'global temperature change potential (GTP100) no LT')

ReCiPe 2016: 256 methods, e.g. ('ReCiPe 2016 v1.03, endpoint (E) no LT', 'ecosystem quality no LT', 'acidification: terrestrial no LT')

EF v3: 146 methods, e.g. ('EF v3.0 no LT', 'acidification no LT', 'accumulated exceedance (AE) no LT')

✅ Environment ready. Next: **01 — the data model & the math of LCA**.